# Nemotron Reasoning Challenge — Submission Notebook

This competition expects a **`/kaggle/working/submission.zip`** containing a
LoRA adapter (max rank 32) trained against `Nemotron-3-Nano-30B-A3B-BF16`.
Kaggle runs *their own* inference using your adapter on the hidden test set.

So this notebook does **just one thing**: package the adapter (attached as a
Kaggle Dataset) into `submission.zip`. No vllm, no model load, no inference.

## Required Kaggle setup

Right-side **Settings** panel:
1. **Accelerator**: GPU L4 ×4 (or any — we don't use it). Internet **OFF**.
2. **Add Data** → attach your LoRA adapter dataset (e.g. `sebmontreal/nemotron-lora-adapter`).
3. Competition data auto-attaches; we don't actually use it here.

## Phase 0 — Locate the LoRA adapter under `/kaggle/input/`

In [ ]:
import json, os, shutil, zipfile
from pathlib import Path

IS_KAGGLE = Path("/kaggle/input").is_dir()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path("_kernel_output")
WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()

MAX_LORA_RANK = 32  # Competition cap.


def _find_adapter_dir(root: Path) -> Path | None:
    """Return the dir under `root` that contains an `adapter_config.json` +
    `adapter_model.safetensors` pair. Handles flat or one-deep-nested layouts."""
    if not root.is_dir():
        return None
    for top in sorted(p for p in root.iterdir() if p.is_dir()):
        if (top / "adapter_config.json").is_file() and any(top.glob("adapter_model.*")):
            return top
        for sub in top.rglob("adapter_config.json"):
            if sub.is_file() and any(sub.parent.glob("adapter_model.*")):
                return sub.parent
    return None


ADAPTER_DIR = _find_adapter_dir(INPUT_ROOT)
if ADAPTER_DIR is None:
    listing = "\n".join(f"  - {p}" for p in sorted(INPUT_ROOT.iterdir())) if INPUT_ROOT.is_dir() else "  (no input root)"
    raise SystemExit(
        "No LoRA adapter found under "
        f"{INPUT_ROOT}. Attach 'sebmontreal/nemotron-lora-adapter' (or similar) "
        f"via 'Add Data' and re-run.\n\nMounted under {INPUT_ROOT}:\n{listing}"
    )

print(f"ADAPTER_DIR = {ADAPTER_DIR}")
for p in sorted(ADAPTER_DIR.iterdir()):
    if p.is_file():
        print(f"  {p.name}  ({p.stat().st_size / 1e6:.1f} MB)")

## Phase 1 — Validate `adapter_config.json` meets competition constraints

The competition caps LoRA `r <= 32`. Fail fast if our adapter exceeds it,
before we waste cycles zipping a non-conformant submission.

In [ ]:
_cfg_path = ADAPTER_DIR / "adapter_config.json"
with _cfg_path.open(encoding="utf-8") as _f:
    _cfg = json.load(_f)

_r = _cfg.get("r")
_alpha = _cfg.get("lora_alpha")
_base = _cfg.get("base_model_name_or_path")
_targets = _cfg.get("target_modules")

print(f"r:              {_r}")
print(f"lora_alpha:     {_alpha}")
print(f"base_model:     {_base}")
print(f"target_modules: {_targets}")

if _r is None or _r > MAX_LORA_RANK:
    raise SystemExit(
        f"LoRA rank r={_r} exceeds competition cap of {MAX_LORA_RANK}. "
        "Retrain with --lora-r 32 or lower before submitting."
    )
print(f"OK: rank {_r} is within the competition cap of {MAX_LORA_RANK}.")

## Phase 2 — Package the adapter into `submission.zip`

Includes the adapter weights + config + tokenizer + chat template so the
graders can apply the adapter exactly as you trained it.

In [ ]:
SUBMISSION_ZIP = WORK_ROOT / "submission.zip"
if SUBMISSION_ZIP.exists():
    SUBMISSION_ZIP.unlink()

_total_bytes = 0
_files_in_zip: list[str] = []
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_STORED) as _zf:
    for src in sorted(ADAPTER_DIR.iterdir()):
        if not src.is_file():
            continue
        # safetensors are already compressed; ZIP_STORED avoids extra CPU and
        # produces a deterministic file size.
        _zf.write(src, arcname=src.name)
        _files_in_zip.append(src.name)
        _total_bytes += src.stat().st_size

_zip_size = SUBMISSION_ZIP.stat().st_size
print(f"Wrote {SUBMISSION_ZIP}")
print(f"  source bytes: {_total_bytes:,} ({_total_bytes / 1e9:.2f} GB)")
print(f"  zip    bytes: {_zip_size:,} ({_zip_size / 1e9:.2f} GB)")
print("  files:")
for name in _files_in_zip:
    print(f"    - {name}")

## Phase 3 — Sanity check the zip

Crack the zip back open and confirm the two critical files
(`adapter_config.json` + `adapter_model.safetensors`) are present and that
the config inside the zip still passes the rank check. This catches subtle
issues (truncated upload, wrong layout) *before* you click Submit.

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "r") as _zf:
    _names = _zf.namelist()
    print("submission.zip contents:")
    for n in _names:
        info = _zf.getinfo(n)
        print(f"  {n}  ({info.file_size / 1e6:.1f} MB)")

    if "adapter_config.json" not in _names:
        raise SystemExit("submission.zip is missing adapter_config.json")
    if not any(n.startswith("adapter_model.") for n in _names):
        raise SystemExit("submission.zip is missing adapter_model.safetensors (or sharded weights)")

    with _zf.open("adapter_config.json") as _f:
        _cfg_in_zip = json.load(_f)
    _r_zip = _cfg_in_zip.get("r")
    if _r_zip is None or _r_zip > MAX_LORA_RANK:
        raise SystemExit(f"adapter_config.json inside zip has rank {_r_zip} > {MAX_LORA_RANK}")
    print(f"OK: adapter_config.json inside zip has rank r={_r_zip}.")

print(f"\nReady to submit. View: ls -lh {SUBMISSION_ZIP}")